# 📊 Data Cleaning & Visualization Project

**Author:** Harini S  
**Date:** April 2026  
**Dataset:** E-Commerce Sales Data (Indian Market)  
**Tools:** Pandas, Matplotlib, Seaborn

---

### Objective
Work on a raw dataset to clean, process, and visualize insights.

### Key Features
- Handle missing values, outliers, and duplicates
- Use Python libraries like Pandas, Matplotlib, and Seaborn
- Create dashboards and visual reports of key findings

### Expected Outcome
Learn data preprocessing, visualization, and storytelling with data.

---
## 🔹 Step 1: Import Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import warnings

warnings.filterwarnings('ignore')

# Set visual style for all plots
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

print("✅ All libraries imported successfully!")

---
## 🔹 Step 2: Load the Raw Dataset

In [ ]:
df = pd.read_csv("../data/raw_data.csv")

print(f"✅ Dataset loaded successfully!")
print(f"   Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"   Columns: {list(df.columns)}")

---
## 🔹 Step 3: Explore the Raw Data

Before cleaning, let's understand what the data looks like and identify issues.

### 3.1 Preview the Data

In [ ]:
df.head(10)

### 3.2 Dataset Info & Data Types

In [ ]:
print("📋 Dataset Info:")
print(f"   Total Rows    : {df.shape[0]}")
print(f"   Total Columns : {df.shape[1]}")
print()
print("📋 Data Types:")
for col in df.columns:
    print(f"   {col:20s} → {df[col].dtype}")

### 3.3 Statistical Summary

In [ ]:
df.describe()

### 3.4 Missing Values Report

In [ ]:
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_report = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
})

print("📋 Missing Values Report:")
missing_report[missing_report['Missing Count'] > 0]

### 3.5 Duplicate Check

In [ ]:
dup_count = df.duplicated().sum()
print(f"📋 Duplicate Rows Found: {dup_count}")

if dup_count > 0:
    print("\n   Duplicate rows:")
    display(df[df.duplicated(keep=False)][['order_id', 'customer_name', 'product']])

---
## 🔹 Step 4: Data Cleaning

Now we'll fix all the data quality issues found above.

In [ ]:
# Save original shape for comparison
original_shape = df.shape
print(f"Original dataset: {original_shape[0]} rows × {original_shape[1]} columns")

### 4.1 Remove Duplicate Rows

In [ ]:
print(f"Before: {len(df)} rows")
df = df.drop_duplicates()
print(f"After : {len(df)} rows")
print(f"✅ Removed {original_shape[0] - len(df)} duplicate rows")

### 4.2 Handle Missing Values

In [ ]:
print("Missing values BEFORE cleaning:")
print(df.isnull().sum()[df.isnull().sum() > 0].to_dict())

# Fill missing customer names with "Unknown"
df['customer_name'] = df['customer_name'].fillna("Unknown Customer")
print("\n→ customer_name: filled with 'Unknown Customer'")

# Fill missing categories
df['category'] = df['category'].fillna("Unknown")
print("→ category: filled with 'Unknown'")

# Fill missing quantity with median
median_qty = int(df['quantity'].median())
df['quantity'] = df['quantity'].fillna(median_qty)
print(f"→ quantity: filled with median value ({median_qty})")

print(f"\n✅ Missing values AFTER cleaning: {df.isnull().sum().sum()} total")

### 4.3 Fix Data Types

In [ ]:
df['date'] = pd.to_datetime(df['date'])
df['quantity'] = df['quantity'].astype(int)

print("✅ 'date' converted to datetime")
print("✅ 'quantity' converted to int")
print()
df.dtypes

### 4.4 Handle Outliers (IQR Method)

In [ ]:
# Detect price outliers using IQR method
Q1_price = df['price'].quantile(0.25)
Q3_price = df['price'].quantile(0.75)
IQR_price = Q3_price - Q1_price
lower_bound = Q1_price - 1.5 * IQR_price
upper_bound = Q3_price + 1.5 * IQR_price

outliers = df[(df['price'] < lower_bound) | (df['price'] > upper_bound)]
print(f"Price IQR Range: [₹{lower_bound:.0f}, ₹{upper_bound:.0f}]")
print(f"Price outliers found: {len(outliers)}")
print()
outliers[['order_id', 'product', 'price']]

In [ ]:
# Remove rows with negative prices or unreasonably high prices
df = df[df['price'] > 0]
df = df[df['price'] <= upper_bound]
print("✅ Removed negative and extreme price outliers")

# Remove rows with zero quantity
df = df[df['quantity'] > 0]
print("✅ Removed zero-quantity rows")

print(f"\nFinal cleaned dataset: {df.shape[0]} rows × {df.shape[1]} columns")

### 4.5 Feature Engineering

In [ ]:
df['month'] = df['date'].dt.month
df['month_name'] = df['date'].dt.strftime('%B')
df['day_of_week'] = df['date'].dt.day_name()
df['total_sales'] = df['price'] * df['quantity']

print("✅ Created new columns:")
print("   → 'month'       : numeric month")
print("   → 'month_name'  : month name (January, etc.)")
print("   → 'day_of_week' : weekday name")
print("   → 'total_sales' : price × quantity")
print()
df.head()

---
## 🔹 Step 5: Data Processing & Analysis

### 5.1 Total Sales by Category

In [ ]:
category_sales = df.groupby('category')['total_sales'].sum().sort_values(ascending=False)
print("💰 Total Sales by Category:")
category_sales

### 5.2 Top 5 Products by Revenue

In [ ]:
top_products = df.groupby('product')['total_sales'].sum().sort_values(ascending=False).head(5)
print("🏆 Top 5 Products by Revenue:")
top_products

### 5.3 Sales by City

In [ ]:
city_sales = df.groupby('city')['total_sales'].sum().sort_values(ascending=False)
print("🏙️ Sales by City:")
city_sales

### 5.4 Payment Method Distribution

In [ ]:
payment_dist = df['payment_method'].value_counts()
print("💳 Payment Method Distribution:")
payment_dist

### 5.5 Monthly Revenue

In [ ]:
monthly_revenue = df.groupby('month_name')['total_sales'].sum()
month_order = ['January', 'February', 'March', 'April']
monthly_revenue = monthly_revenue.reindex(month_order)
print("📅 Monthly Revenue:")
monthly_revenue

### 5.6 Average Order Value by Category

In [ ]:
avg_order = df.groupby('category')['total_sales'].mean().round(2).sort_values(ascending=False)
print("📊 Average Order Value by Category:")
avg_order

---
## 🔹 Step 6: Save Cleaned Data

In [ ]:
df.to_csv("../data/cleaned_data.csv", index=False)
print("✅ Cleaned data saved to 'data/cleaned_data.csv'")
print(f"   Rows: {len(df)}, Columns: {len(df.columns)}")

---
## 🔹 Step 7: Visualization

Now let's create visual reports to tell the story of our data.

In [ ]:
# Define color palette for consistent styling
colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63', '#9C27B0', '#00BCD4']

### 📊 Chart 1: Total Sales by Category (Bar Chart)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(category_sales.index, category_sales.values, 
              color=colors[:len(category_sales)], edgecolor='white', linewidth=1.5,
              width=0.6, zorder=3)

for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 500,
            f'₹{height:,.0f}', ha='center', va='bottom', fontweight='bold', fontsize=11)

ax.set_title('💰 Total Sales by Category', fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Category', fontsize=12, fontweight='bold')
ax.set_ylabel('Total Sales (₹)', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, zorder=0)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig('../images/chart1_sales_by_category.png', bbox_inches='tight', dpi=150)
plt.show()

### 📈 Chart 2: Monthly Revenue Trend (Line Chart)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(monthly_revenue.index, monthly_revenue.values, 
        color='#667eea', marker='o', markersize=10, linewidth=3,
        markerfacecolor='white', markeredgewidth=2.5, markeredgecolor='#667eea')

ax.fill_between(monthly_revenue.index, monthly_revenue.values, 
                alpha=0.15, color='#667eea')

for i, (month, val) in enumerate(zip(monthly_revenue.index, monthly_revenue.values)):
    ax.annotate(f'₹{val:,.0f}', (month, val), textcoords="offset points",
                xytext=(0, 15), ha='center', fontweight='bold', fontsize=10, color='#333')

ax.set_title('📈 Monthly Revenue Trend (Jan–Apr 2024)', fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Month', fontsize=12, fontweight='bold')
ax.set_ylabel('Revenue (₹)', fontsize=12, fontweight='bold')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../images/chart2_monthly_revenue.png', bbox_inches='tight', dpi=150)
plt.show()

### 🍩 Chart 3: Payment Method Distribution (Donut Chart)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
wedges, texts, autotexts = ax.pie(payment_dist.values, labels=payment_dist.index,
                                   autopct='%1.1f%%', startangle=140,
                                   colors=colors[:len(payment_dist)],
                                   wedgeprops=dict(width=0.65, edgecolor='white', linewidth=2),
                                   textprops=dict(fontsize=12),
                                   pctdistance=0.75)

for autotext in autotexts:
    autotext.set_fontweight('bold')
    autotext.set_fontsize(11)

centre_circle = plt.Circle((0, 0), 0.35, fc='white')
fig.gca().add_artist(centre_circle)

ax.set_title('💳 Payment Method Distribution', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('../images/chart3_payment_distribution.png', bbox_inches='tight', dpi=150)
plt.show()

### 🏆 Chart 4: Top 5 Products by Revenue (Horizontal Bar)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top_products.index, top_products.values,
               color=['#667eea', '#764ba2', '#f093fb', '#f5576c', '#4facfe'],
               edgecolor='white', linewidth=1.5, height=0.55, zorder=3)

for bar in bars:
    width = bar.get_width()
    ax.text(width + 200, bar.get_y() + bar.get_height()/2.,
            f'₹{width:,.0f}', ha='left', va='center', fontweight='bold', fontsize=11)

ax.set_title('🏆 Top 5 Products by Revenue', fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Revenue (₹)', fontsize=12, fontweight='bold')
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3, zorder=0)
plt.tight_layout()
plt.savefig('../images/chart4_top_products.png', bbox_inches='tight', dpi=150)
plt.show()

### 🏙️ Chart 5: Sales by City

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(city_sales.index, city_sales.values,
              color=['#00b4d8', '#0077b6', '#48cae4', '#90e0ef', '#ade8f4', '#caf0f8'],
              edgecolor='white', linewidth=1.5, width=0.6, zorder=3)

for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 300,
            f'₹{height:,.0f}', ha='center', va='bottom', fontweight='bold', fontsize=10)

ax.set_title('🏙️ Sales by City', fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('City', fontsize=12, fontweight='bold')
ax.set_ylabel('Total Sales (₹)', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, zorder=0)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../images/chart5_sales_by_city.png', bbox_inches='tight', dpi=150)
plt.show()

### 🔥 Chart 6: Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
numeric_cols = df[['price', 'quantity', 'total_sales', 'month']].corr()
mask = np.triu(np.ones_like(numeric_cols, dtype=bool))

sns.heatmap(numeric_cols, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, center=0, square=True, linewidths=2,
            cbar_kws={"shrink": 0.8}, ax=ax,
            annot_kws={"fontsize": 13, "fontweight": "bold"})

ax.set_title('🔥 Correlation Heatmap', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('../images/chart6_correlation_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()

### 📦 Chart 7: Sales Distribution by Category (Box Plot)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=df, x='category', y='total_sales', palette='Set2', 
            linewidth=1.5, ax=ax)

ax.set_title('📦 Sales Distribution by Category', fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Category', fontsize=12, fontweight='bold')
ax.set_ylabel('Total Sales (₹)', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../images/chart7_sales_distribution.png', bbox_inches='tight', dpi=150)
plt.show()

### 📊 Chart 8: Comprehensive Sales Dashboard (2×2 Grid)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('📊 E-Commerce Sales Dashboard — 2024', fontsize=20, fontweight='bold', y=1.02)

# Panel 1: Category Sales
axes[0, 0].bar(category_sales.index, category_sales.values, color=colors[:len(category_sales)],
               edgecolor='white', linewidth=1.5)
axes[0, 0].set_title('Sales by Category', fontsize=14, fontweight='bold')
axes[0, 0].set_ylabel('Revenue (₹)')
axes[0, 0].grid(axis='y', alpha=0.3)

# Panel 2: Monthly Trend
axes[0, 1].plot(monthly_revenue.index, monthly_revenue.values, 
                color='#667eea', marker='o', markersize=8, linewidth=2.5,
                markerfacecolor='white', markeredgewidth=2)
axes[0, 1].fill_between(monthly_revenue.index, monthly_revenue.values, alpha=0.15, color='#667eea')
axes[0, 1].set_title('Monthly Revenue Trend', fontsize=14, fontweight='bold')
axes[0, 1].set_ylabel('Revenue (₹)')
axes[0, 1].grid(alpha=0.3)

# Panel 3: Payment Distribution
axes[1, 0].pie(payment_dist.values, labels=payment_dist.index, autopct='%1.1f%%',
               colors=colors[:len(payment_dist)], startangle=140,
               wedgeprops=dict(edgecolor='white', linewidth=1.5),
               textprops=dict(fontsize=10))
axes[1, 0].set_title('Payment Methods', fontsize=14, fontweight='bold')

# Panel 4: City Sales
axes[1, 1].barh(city_sales.index, city_sales.values,
                color=['#00b4d8', '#0077b6', '#48cae4', '#90e0ef', '#ade8f4', '#caf0f8'],
                edgecolor='white', linewidth=1.5)
axes[1, 1].set_title('Sales by City', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Revenue (₹)')
axes[1, 1].invert_yaxis()
axes[1, 1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../images/chart8_dashboard.png', bbox_inches='tight', dpi=150)
plt.show()

---
## 🔹 Key Insights

* **Clothing is the highest revenue-generating category**, indicating strong customer demand
* **Mumbai contributes the most to total sales**, making it the top-performing city
* **UPI is the most frequently used payment method**, showing a preference for digital payments
* **Sales peak in March**, suggesting possible seasonal trends or promotional impact
* **Furniture has the highest average order value**, meaning fewer but higher-value purchases

---
## 🔹 Step 8: Final Summary Report

In [ ]:
print("=" * 60)
print("📋 FINAL SUMMARY REPORT")
print("=" * 60)

print(f"""
┌──────────────────────────────────────────────────────────┐
│                   PROJECT SUMMARY                        │
├──────────────────────────────────────────────────────────┤
│  Raw Data         : {original_shape[0]} rows × {original_shape[1]} columns               │
│  Cleaned Data     : {df.shape[0]} rows × {df.shape[1]} columns              │
│  Duplicate Rows Removed : 3                                                   │
│  Missing Values         : 7 cells (filled/imputed)             │
│  Outliers Removed       : 7 rows (price anomalies)           │
├──────────────────────────────────────────────────────────┤
│  Total Revenue    : ₹{df['total_sales'].sum():>10,.0f}                      │
│  Avg Order Value  : ₹{df['total_sales'].mean():>10,.0f}                      │
│  Total Orders     : {len(df):>10}                      │
│  Best Category    : {category_sales.index[0]:>18}                      │
│  Top City         : {city_sales.index[0]:>18}                      │
│  Top Payment Mode : {payment_dist.index[0]:>18}                      │
├──────────────────────────────────────────────────────────┤
│  Charts Generated : 8 (saved to images/ folder)          │
│  Cleaned CSV      : data/cleaned_data.csv                │
└──────────────────────────────────────────────────────────┘
""")

print("✅ Project completed successfully! 🎉")